# Разведочный анализ данных (EDA) — Fashionpedia

Ноутбук рассчитан на запуск в **Kaggle Notebooks** с включёнными **GPU** и **Internet**.

Загрузка данных выполняется той же функцией, что и в `main.py --prepare-data` (`src/dataset/dataset.py`), что обеспечивает согласованность подготовки данных.

Содержание:
1. Загрузка датасета
2. Структура и размеры выборок
3. Распределение объектов по категориям
4. Число объектов на изображение
5. Распределение размеров bounding box
6. Визуализация изображений с рамками

## 0. Подготовка окружения

В Kaggle добавьте корень репозитория в `sys.path`, чтобы импортировать `src`. Скорректируйте путь под расположение проекта в вашей сессии.

In [ ]:
import sys
from pathlib import Path

# Путь к корню репозитория (скорректируйте при необходимости).
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

## 1. Загрузка датасета

Используем ту же функцию загрузки, что и в основном пайплайне. Для первичного EDA удобно работать без разбиения train/test — загружаем официальные split-ы `train` и `val`.

При ограниченном диске задайте `subset_size` (например, 2000) или включите `streaming=True`.

In [ ]:
from src.dataset.dataset import load_fashionpedia, get_category_names

dataset = load_fashionpedia(streaming=False, subset_size=None)
category_names = get_category_names(dataset['train'])
num_categories = len(category_names)
print(f'Число категорий: {num_categories}')

## 2. Структура и размеры выборок

In [ ]:
print('Split-ы:', list(dataset.keys()))
print('train:', len(dataset['train']), 'изображений')
print('val:  ', len(dataset['val']), 'изображений')
print('\nСтруктура одного примера:')
example = dataset['train'][0]
print('Ключи:', list(example.keys()))
print('Ключи objects:', list(example['objects'].keys()))
print('Число объектов на первом изображении:', len(example['objects']['category']))

## 3. Распределение объектов по категориям

Подсчитываем число экземпляров каждой категории в обучающей выборке. Результат используется для отбора 10–13 наиболее представленных категорий и для оценки дисбаланса классов.

In [ ]:
category_counter = Counter()
for objects in dataset['train']['objects']:
    category_counter.update(objects['category'])

counts_df = (
    pd.DataFrame(
        {'category_id': list(category_counter.keys()),
         'count': list(category_counter.values())}
    )
    .assign(category=lambda d: d['category_id'].map(lambda i: category_names[i]))
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
counts_df.head(20)

In [ ]:
top_n = 20
plot_df = counts_df.head(top_n)
plt.figure(figsize=(12, 6))
plt.bar(plot_df['category'], plot_df['count'])
plt.xticks(rotation=75, ha='right')
plt.ylabel('Число экземпляров')
plt.title(f'Распределение объектов по категориям (топ-{top_n})')
plt.tight_layout()
plt.show()

## 4. Число объектов на изображение

In [ ]:
objects_per_image = np.array([len(o['category']) for o in dataset['train']['objects']])
print(f'Среднее число объектов на изображение: {objects_per_image.mean():.2f}')
print(f'Минимум: {objects_per_image.min()}  Максимум: {objects_per_image.max()}')
print(f'Медиана: {np.median(objects_per_image):.1f}')

plt.figure(figsize=(10, 5))
plt.hist(objects_per_image, bins=range(0, objects_per_image.max() + 2))
plt.xlabel('Число объектов на изображении')
plt.ylabel('Число изображений')
plt.title('Распределение числа объектов на изображение')
plt.tight_layout()
plt.show()

## 5. Распределение размеров bounding box

Градация размеров по методике COCO: малые (площадь < 32²), средние (32²–96²), крупные (> 96²).

In [ ]:
areas = []
for objects in dataset['train']['objects']:
    for box in objects['bbox']:
        x_min, y_min, x_max, y_max = box
        areas.append((x_max - x_min) * (y_max - y_min))
areas = np.array(areas, dtype=float)

small = int((areas < 32 ** 2).sum())
medium = int(((areas >= 32 ** 2) & (areas < 96 ** 2)).sum())
large = int((areas >= 96 ** 2).sum())
total = len(areas)
print(f'Малые объекты:   {small} ({100 * small / total:.1f}%)')
print(f'Средние объекты: {medium} ({100 * medium / total:.1f}%)')
print(f'Крупные объекты: {large} ({100 * large / total:.1f}%)')

plt.figure(figsize=(8, 5))
plt.bar(['малые', 'средние', 'крупные'], [small, medium, large])
plt.ylabel('Число объектов')
plt.title('Распределение размеров bounding box (градация COCO)')
plt.tight_layout()
plt.show()

## 6. Визуализация изображений с рамками

In [ ]:
from src.utils.utils import draw_boxes

num_samples = 4
fig, axes = plt.subplots(1, num_samples, figsize=(5 * num_samples, 5))
for ax, example in zip(axes, dataset['train'].select(range(num_samples))):
    draw_boxes(
        example['image'],
        example['objects']['bbox'],
        labels=example['objects']['category'],
        category_names=category_names,
        ax=ax,
    )
plt.tight_layout()
plt.show()